# EE 451: Communications Systems
## Lesson 22 - Exam Review & Probability

### Learning Objectives
By the end of this lesson, you will be able to:
- Define probability axioms and basic concepts
- Apply conditional probability and Bayes' theorem
- Calculate probabilities for independent events
- Understand random experiments and sample spaces in communications context
- Compute packet error rates from bit error rates

### Textbook Reference
Haykin & Moher, Chapter 8.1-8.2

In [ ]:
# Setup: Import required libraries
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Use consistent plot style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete! NumPy version:", np.__version__)

## Part 1: Probability Axioms & Sample Spaces

**Random Experiment:** Outcome is not deterministic

**Sample Space $S$:** Set of all possible outcomes
- Coin flip: $S = \{H, T\}$
- Die roll: $S = \{1, 2, 3, 4, 5, 6\}$
- Bit transmission: $S = \{0, 1\}$

**Probability Axioms:**
1. **Non-negativity:** $0 \leq P(A) \leq 1$
2. **Normalization:** $P(S) = 1$
3. **Additivity:** $P(A \cup B) = P(A) + P(B)$ if $A$ and $B$ are mutually exclusive

**General Addition Rule:** $P(A \cup B) = P(A) + P(B) - P(A \cap B)$

In [ ]:
# Part 1: Probability Axioms - Simulation vs Theory

np.random.seed(42)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Law of Large Numbers (coin flip) ---
ax = axes[0]
N = 10000
flips = np.random.choice([0, 1], size=N)  # 0=Tails, 1=Heads
cumulative_prob = np.cumsum(flips) / np.arange(1, N + 1)
ax.plot(np.arange(1, N + 1), cumulative_prob, 'b-', alpha=0.8)
ax.axhline(y=0.5, color='r', linestyle='--', linewidth=2, label='P(H) = 0.5 (theory)')
ax.set_xlabel('Number of Flips')
ax.set_ylabel('Relative Frequency of Heads')
ax.set_title('Law of Large Numbers: Coin Flip')
ax.set_xscale('log')
ax.legend()

# --- Plot 2: Die roll PMF (simulated vs theoretical) ---
ax = axes[1]
N_rolls = 10000
rolls = np.random.randint(1, 7, size=N_rolls)
values, counts = np.unique(rolls, return_counts=True)
empirical_pmf = counts / N_rolls
theoretical_pmf = np.ones(6) / 6

x = np.arange(1, 7)
width = 0.35
ax.bar(x - width/2, empirical_pmf, width, label=f'Empirical (N={N_rolls:,})', alpha=0.7)
ax.bar(x + width/2, theoretical_pmf, width, label='Theoretical (1/6)', alpha=0.7)
ax.set_xlabel('Die Face')
ax.set_ylabel('Probability')
ax.set_title('Die Roll: Empirical vs Theoretical PMF')
ax.set_xticks(x)
ax.legend()

# --- Plot 3: Additivity axiom verification ---
ax = axes[2]
# Mutually exclusive events: A = {1,2}, B = {3,4}
A_count = np.sum((rolls == 1) | (rolls == 2))
B_count = np.sum((rolls == 3) | (rolls == 4))
AuB_count = np.sum((rolls == 1) | (rolls == 2) | (rolls == 3) | (rolls == 4))

P_A = A_count / N_rolls
P_B = B_count / N_rolls
P_AuB = AuB_count / N_rolls

labels = ['P(A)', 'P(B)', 'P(A)+P(B)', 'P(A\u222aB)']
values_plot = [P_A, P_B, P_A + P_B, P_AuB]
colors = ['steelblue', 'coral', 'mediumpurple', 'green']
bars = ax.bar(labels, values_plot, color=colors, alpha=0.7)
ax.set_ylabel('Probability')
ax.set_title('Additivity: P(A\u222aB) = P(A) + P(B)\n(A={1,2}, B={3,4}, mutually exclusive)')
for bar, val in zip(bars, values_plot):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Print verification
print("=" * 55)
print("Probability Axiom Verification (Die Roll Experiment)")
print("=" * 55)
print(f"Total rolls: {N_rolls:,}")
print(f"\nAxiom 1 (Non-negativity): All P(x) >= 0  (verified)")
print(f"Axiom 2 (Normalization): Sum of PMF = {np.sum(empirical_pmf):.6f} (approx 1)")
print(f"Axiom 3 (Additivity): P(A) + P(B) = {P_A + P_B:.4f}, P(A union B) = {P_AuB:.4f}")
print(f"  Difference: {abs(P_A + P_B - P_AuB):.6f} (approx 0 for mutually exclusive)")

## Part 2: Conditional Probability

**Definition:** $P(A|B) = \frac{P(A \cap B)}{P(B)}$

"Probability of $A$ given that $B$ has occurred"

**Binary Symmetric Channel (BSC):**
- Crossover probability $p$: probability a bit is flipped
- $P(\text{Recv } 0 \mid \text{Sent } 0) = 1 - p$
- $P(\text{Recv } 1 \mid \text{Sent } 0) = p$
- $P(\text{Recv } 0 \mid \text{Sent } 1) = p$
- $P(\text{Recv } 1 \mid \text{Sent } 1) = 1 - p$

In [ ]:
# Part 2: Conditional Probability - Binary Symmetric Channel

p_error = 0.1  # Crossover probability
N = 100000
np.random.seed(42)

# Transmitted bits: P(send 0) = P(send 1) = 0.5
tx_bits = np.random.randint(0, 2, size=N)

# Channel introduces errors with probability p_error
errors = np.random.random(size=N) < p_error
rx_bits = tx_bits ^ errors.astype(int)  # XOR introduces bit flips

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: BSC transition diagram ---
ax = axes[0]
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Binary Symmetric Channel (BSC)', fontsize=14, fontweight='bold')

# Draw nodes
circle_props = dict(facecolor='lightblue', edgecolor='black', linewidth=2)
ax.add_patch(plt.Circle((2, 7), 0.6, **circle_props))
ax.text(2, 7, '0', ha='center', va='center', fontsize=16, fontweight='bold')
ax.add_patch(plt.Circle((2, 3), 0.6, **circle_props))
ax.text(2, 3, '1', ha='center', va='center', fontsize=16, fontweight='bold')

circle_props_rx = dict(facecolor='lightyellow', edgecolor='black', linewidth=2)
ax.add_patch(plt.Circle((8, 7), 0.6, **circle_props_rx))
ax.text(8, 7, '0', ha='center', va='center', fontsize=16, fontweight='bold')
ax.add_patch(plt.Circle((8, 3), 0.6, **circle_props_rx))
ax.text(8, 3, '1', ha='center', va='center', fontsize=16, fontweight='bold')

ax.text(2, 9, 'Sent', ha='center', fontsize=12, fontstyle='italic')
ax.text(8, 9, 'Received', ha='center', fontsize=12, fontstyle='italic')

# Draw arrows
arrow_props = dict(arrowstyle='->', linewidth=2)
ax.annotate('', xy=(7.4, 7), xytext=(2.6, 7),
            arrowprops=dict(**arrow_props, color='green'))
ax.text(5, 7.5, f'1-p = {1-p_error}', ha='center', fontsize=11, color='green')
ax.annotate('', xy=(7.4, 3), xytext=(2.6, 3),
            arrowprops=dict(**arrow_props, color='green'))
ax.text(5, 2.5, f'1-p = {1-p_error}', ha='center', fontsize=11, color='green')
ax.annotate('', xy=(7.4, 3), xytext=(2.6, 7),
            arrowprops=dict(**arrow_props, color='red', linestyle='dashed'))
ax.text(5.5, 5.5, f'p = {p_error}', ha='center', fontsize=11, color='red')
ax.annotate('', xy=(7.4, 7), xytext=(2.6, 3),
            arrowprops=dict(**arrow_props, color='red', linestyle='dashed'))
ax.text(4.5, 4.5, f'p = {p_error}', ha='center', fontsize=11, color='red')

# --- Plot 2: Confusion matrix ---
ax = axes[1]
sent_0, sent_1 = tx_bits == 0, tx_bits == 1
recv_0, recv_1 = rx_bits == 0, rx_bits == 1

P_r0_s0 = np.sum(recv_0 & sent_0) / np.sum(sent_0)
P_r1_s0 = np.sum(recv_1 & sent_0) / np.sum(sent_0)
P_r0_s1 = np.sum(recv_0 & sent_1) / np.sum(sent_1)
P_r1_s1 = np.sum(recv_1 & sent_1) / np.sum(sent_1)

confusion = np.array([[P_r0_s0, P_r1_s0], [P_r0_s1, P_r1_s1]])
im = ax.imshow(confusion, cmap='Blues', vmin=0, vmax=1)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Received 0', 'Received 1'])
ax.set_yticklabels(['Sent 0', 'Sent 1'])
ax.set_title('Conditional Probability Matrix')

for i in range(2):
    for j in range(2):
        color = 'white' if confusion[i, j] > 0.5 else 'black'
        ax.text(j, i, f'{confusion[i,j]:.4f}', ha='center', va='center',
                fontsize=14, color=color, fontweight='bold')

plt.colorbar(im, ax=ax, label='Probability')
plt.tight_layout()
plt.show()

# Print summary
print("=" * 55)
print("Conditional Probability Results (BSC)")
print("=" * 55)
print(f"Channel error probability: p = {p_error}")
print(f"Number of transmissions: N = {N:,}")
print(f"\nP(Recv 0 | Sent 0) = {P_r0_s0:.4f}  (theory: {1-p_error})")
print(f"P(Recv 1 | Sent 0) = {P_r1_s0:.4f}  (theory: {p_error})")
print(f"P(Recv 0 | Sent 1) = {P_r0_s1:.4f}  (theory: {p_error})")
print(f"P(Recv 1 | Sent 1) = {P_r1_s1:.4f}  (theory: {1-p_error})")

## Part 3: Bayes' Theorem

**Bayes' Theorem:** $P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$

Reverses the conditional: know $P(B|A)$, want $P(A|B)$

**Application:** Given a received bit, what was most likely sent?
- **Prior:** $P(\text{Sent} = 0)$, $P(\text{Sent} = 1)$
- **Likelihood:** $P(\text{Recv} | \text{Sent})$ from BSC model
- **Posterior:** $P(\text{Sent} | \text{Recv})$ via Bayes

**Total Probability:** $P(B) = \sum_i P(B|A_i) P(A_i)$

In [ ]:
# Part 3: Bayes' Theorem - Inferring Transmitted Bit

# Unequal priors to make Bayes' effect visible
P_sent0 = 0.6
P_sent1 = 0.4
p_e = 0.1  # Channel error probability

# Likelihoods (from BSC model)
P_r0_s0 = 1 - p_e  # P(Recv=0 | Sent=0)
P_r1_s0 = p_e       # P(Recv=1 | Sent=0)
P_r0_s1 = p_e       # P(Recv=0 | Sent=1)
P_r1_s1 = 1 - p_e   # P(Recv=1 | Sent=1)

# Total probabilities
P_recv0 = P_r0_s0 * P_sent0 + P_r0_s1 * P_sent1
P_recv1 = P_r1_s0 * P_sent0 + P_r1_s1 * P_sent1

# Posterior probabilities (Bayes' theorem)
P_s0_r0 = P_r0_s0 * P_sent0 / P_recv0
P_s1_r0 = P_r0_s1 * P_sent1 / P_recv0
P_s0_r1 = P_r1_s0 * P_sent0 / P_recv1
P_s1_r1 = P_r1_s1 * P_sent1 / P_recv1

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Prior probabilities
ax = axes[0]
bars = ax.bar(['P(Sent=0)', 'P(Sent=1)'], [P_sent0, P_sent1],
              color=['steelblue', 'coral'], alpha=0.8)
ax.set_ylabel('Probability')
ax.set_title('Prior Probabilities')
ax.set_ylim(0, 1)
for bar, val in zip(bars, [P_sent0, P_sent1]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=12, fontweight='bold')

# Plot 2: Posterior given Received=0
ax = axes[1]
bars = ax.bar(['P(S=0|R=0)', 'P(S=1|R=0)'], [P_s0_r0, P_s1_r0],
              color=['steelblue', 'coral'], alpha=0.8)
ax.set_ylabel('Probability')
ax.set_title('Posterior: Given Received = 0')
ax.set_ylim(0, 1)
for bar, val in zip(bars, [P_s0_r0, P_s1_r0]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

# Plot 3: Posterior given Received=1
ax = axes[2]
bars = ax.bar(['P(S=0|R=1)', 'P(S=1|R=1)'], [P_s0_r1, P_s1_r1],
              color=['steelblue', 'coral'], alpha=0.8)
ax.set_ylabel('Probability')
ax.set_title('Posterior: Given Received = 1')
ax.set_ylim(0, 1)
for bar, val in zip(bars, [P_s0_r1, P_s1_r1]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.4f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# Print Bayes' theorem worked example
print("=" * 60)
print("Bayes' Theorem: Binary Channel Inference")
print("=" * 60)
print(f"\nGiven: P(Sent=0) = {P_sent0}, P(Sent=1) = {P_sent1}")
print(f"       Channel error probability p = {p_e}")
print(f"\nStep 1: Total probability of receiving 0")
print(f"  P(R=0) = P(R=0|S=0)P(S=0) + P(R=0|S=1)P(S=1)")
print(f"         = {P_r0_s0}*{P_sent0} + {P_r0_s1}*{P_sent1}")
print(f"         = {P_recv0:.4f}")
print(f"\nStep 2: Apply Bayes' theorem")
print(f"  P(S=0 | R=0) = P(R=0|S=0) * P(S=0) / P(R=0)")
print(f"               = {P_r0_s0} * {P_sent0} / {P_recv0:.4f}")
print(f"               = {P_s0_r0:.4f}")
print(f"\n  P(S=1 | R=0) = {P_s1_r0:.4f}")
print(f"\nInterpretation: If we receive '0', there is a {P_s0_r0:.1%}")
print(f"probability that '0' was actually sent.")

## Part 4: Independent Events

**Definition:** Events $A$ and $B$ are **independent** if:
$$P(A \cap B) = P(A) \cdot P(B)$$

Equivalently: $P(A|B) = P(A)$ — knowing $B$ gives no information about $A$

**Communications Application:**
- Noise samples in AWGN channels are independent
- Bit errors in memoryless channels are independent
- For $N$ independent bits with BER $= p$:
  - $P(\text{all correct}) = (1-p)^N$
  - $P(\text{at least one error}) = 1 - (1-p)^N$

In [ ]:
# Part 4: Independent Events - Bit Error Simulation

np.random.seed(42)
N_bits = 100000
p_error = 0.01  # BER = 10^-2

# Simulate independent bit errors
errors = np.random.random(size=N_bits) < p_error

# Verify independence: P(err_n AND err_{n+1}) should = P(err)^2
err_current = errors[:-1]
err_next = errors[1:]

P_err = np.mean(errors)
P_both = np.mean(err_current & err_next)
P_product = P_err ** 2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: Independence verification ---
ax = axes[0]
labels = ['P(Error)', 'P(Error)\u00b2', 'P(Err_n \u2229 Err_{n+1})']
values = [P_err, P_product, P_both]
colors = ['steelblue', 'mediumpurple', 'coral']
bars = ax.bar(labels, values, color=colors, alpha=0.7)
ax.set_ylabel('Probability')
ax.set_title('Independence Test: P(A\u2229B) \u2248 P(A)\u00d7P(B)?')
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.00002,
            f'{val:.6f}', ha='center', fontsize=10, fontweight='bold')

# --- Plot 2: Multiple independent bits ---
ax = axes[1]
n_bits_pkt = np.arange(1, 51)
P_all_correct = (1 - p_error) ** n_bits_pkt
P_at_least_one = 1 - P_all_correct

ax.plot(n_bits_pkt, P_all_correct, 'b-', linewidth=2, label='P(all correct)')
ax.plot(n_bits_pkt, P_at_least_one, 'r--', linewidth=2, label='P(\u2265 1 error)')
ax.set_xlabel('Number of Bits (N)')
ax.set_ylabel('Probability')
ax.set_title(f'Independent Bit Errors (BER = {p_error})')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

# Print verification
print("=" * 55)
print("Independence Verification")
print("=" * 55)
print(f"BER = {p_error}, N = {N_bits:,} bits")
print(f"\nP(Error) = {P_err:.6f}")
print(f"P(Error)^2 = {P_product:.6f}")
print(f"P(Err_n AND Err_{{n+1}}) = {P_both:.6f}")
print(f"Ratio: {P_both/(P_product + 1e-15):.4f} (should be approx 1.0 if independent)")

## Part 5: Communications Application — Packet Error Rate

**Packet of $L$ bits**, each with independent BER $= p$

**Packet Error Rate (PER):** Probability that at least one bit is wrong
$$\text{PER} = 1 - (1 - p)^L$$

**Automatic Repeat Request (ARQ):** Retransmit if packet has error
- After $N$ attempts: $P(\text{success}) = 1 - \text{PER}^N$

In [ ]:
# Part 5: Communications Application - Packet Error Rate

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Plot 1: PER vs BER for various packet sizes ---
ax = axes[0]
BER_values = np.logspace(-6, -1, 200)
packet_sizes = [100, 500, 1000, 5000, 10000]
colors = plt.cm.viridis(np.linspace(0, 0.9, len(packet_sizes)))

for L, color in zip(packet_sizes, colors):
    PER = 1 - (1 - BER_values) ** L
    ax.semilogx(BER_values, PER, linewidth=2, color=color, label=f'L = {L:,} bits')

ax.set_xlabel('Bit Error Rate (BER)')
ax.set_ylabel('Packet Error Rate (PER)')
ax.set_title('Packet Error Rate vs BER')
ax.legend()
ax.set_ylim(0, 1.05)

# --- Plot 2: Reliability through retransmission ---
ax = axes[1]
p_pkt_error = 0.1  # 10% PER
N_tx = np.arange(1, 11)
P_success = 1 - p_pkt_error ** N_tx

ax.bar(N_tx, P_success, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Transmission Attempts (N)')
ax.set_ylabel('P(at least one success)')
ax.set_title(f'Reliability via Retransmission (PER = {p_pkt_error})')
ax.set_xticks(N_tx)
ax.set_ylim(0.85, 1.005)

for n, p in zip(N_tx, P_success):
    ax.text(n, p + 0.002, f'{p:.4f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Print summary
print("=" * 60)
print("Packet Error Rate Examples")
print("=" * 60)
L = 1000
for p in [1e-3, 1e-4, 1e-6]:
    PER = 1 - (1 - p) ** L
    print(f"\n{L}-bit packet, BER = {p:.0e}:")
    print(f"  PER = 1 - (1 - {p})^{L} = {PER:.6f} ({PER:.2%})")

print(f"\nRetransmission (ARQ) with PER = 10%:")
for N in [1, 2, 3, 5]:
    P_s = 1 - 0.1**N
    print(f"  {N} attempt(s): P(success) = {P_s:.6f}")

## Summary

### Key Formulas

| Concept | Formula |
|---------|--------|
| Additivity (mutually exclusive) | $P(A \cup B) = P(A) + P(B)$ |
| General addition rule | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ |
| Conditional probability | $P(A|B) = P(A \cap B) / P(B)$ |
| Bayes' theorem | $P(A|B) = P(B|A) P(A) / P(B)$ |
| Independence | $P(A \cap B) = P(A) \cdot P(B)$ |
| Packet error rate | $\text{PER} = 1 - (1-p)^L$ |
| ARQ reliability | $P(\text{success}) = 1 - \text{PER}^N$ |

### Key Takeaways
1. **Probability axioms** provide the foundation for all BER and noise analysis
2. **Conditional probability** describes how channel errors relate sent and received bits
3. **Bayes' theorem** lets us infer what was transmitted given what we received
4. **Independence** simplifies multi-bit error calculations in memoryless channels
5. **Packet error rate** grows quickly with packet length — even small BER matters for long packets

### Next Topics
- **Lesson 23:** Channel capacity, Shannon-Hartley theorem
- **Lesson 24:** Random variables, Gaussian distribution, Q-function